# Step-by-step implementation
The following are the steps to implement the Multi-Representation Indexing:
  1. Import necessary modules
  2. Set up the OpenAI API key
  3. Load documents and split text
  4. Generate document summaries with LLM
  5. Index with multi-representations
  6. Retrieve documents based on query


## What is Multi-Representation Indexing?

**Multi-Representation Indexing** decouples *what you search over* from *what you return*.

Standard RAG embeds and searches the same raw document chunks you ultimately feed to the
LLM. That's a problem when the raw chunks are long or noisy — a large chunk of text often
doesn't embed as a very sharp semantic vector, so similarity search over full documents
retrieves less precisely.

The fix: index a **compact representation** (an LLM-generated summary) for retrieval, while
keeping the **full original document** around to actually return once that summary is
matched. The two representations are linked by a shared `doc_id`.

- **Vectorstore** (`Chroma`) — embeds and searches only the *summaries* → fast, precise
  semantic matching.
- **Docstore** (`InMemoryByteStore`) — holds the *full original documents*, keyed by
  `doc_id` → nothing is lost for generation.
- **`MultiVectorRetriever`** — ties the two together: it searches the vectorstore, reads off
  the matched summary's `doc_id`, and uses that to fetch the corresponding full document
  from the docstore.

![Multi-Representation Indexing diagram](../assets/img/multi_representation_indexing.png)

**Flow in this notebook:**
1. Load & split source documents (`data/sample/*.txt`).
2. Summarize each document chunk with an LLM.
3. Embed the *summaries* into `Chroma`; store the *full documents* in `InMemoryByteStore`,
   both tagged with the same `doc_id`.
4. At query time, `MultiVectorRetriever` searches the summary embeddings, then returns the
   matching full document(s) — not the summary itself.

## 1. Import necessary modules


In [ ]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.stores import InMemoryByteStore
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
import uuid
from langchain_core.documents import Document

## 2. Set up the OpenAI API key

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

In [ ]:
os.environ["OPENAI_API_KEY"]

In [ ]:
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
if OPENAI_API_KEY == "":
    raise ValueError("Please set the OPENAI_API_KEY environment variable")

## 3. Load documents and split text


In [ ]:
loaders = [
    TextLoader("../../../data/sample/blog.langchain.dev_announcing-langsmith_.txt", encoding="utf-8"),
    TextLoader("../../../data/sample/blog.langchain.dev_automating-web-research_.txt", encoding="utf-8"),
]

docs = []
for loader in loaders:
    docs.extend(loader.load())
text_splitter = RecursiveCharacterTextSplitter(chunk_size=10000)
docs = text_splitter.split_documents(docs)

In [ ]:
len(docs)

## 4. Generate document summaries with LLM


In [ ]:
chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_template("Summarize the following document:\n\n{doc}")
    | ChatOpenAI(model="gpt-3.5-turbo", max_retries=0)
    | StrOutputParser()
)

In [ ]:
summaries = chain.batch(docs, {"max_concurrency": 3})

In [ ]:
summaries

## 5. Index with multi-representations


**Set up the two stores and the retriever that links them:**
- `vectorstore` — `Chroma` collection that will hold **summary embeddings** only.
- `store` — `InMemoryByteStore` acting as the **docstore** for full original documents.
- `id_key = "doc_id"` — the metadata field name used to link a summary to its full document.
- `retriever` — `MultiVectorRetriever` wired to both stores via `id_key`.
- `doc_ids` — one UUID per document chunk, to be shared between its summary and its full text.

In [ ]:
vectorstore = Chroma(collection_name="summaries", embedding_function=OpenAIEmbeddings())

store = InMemoryByteStore()
id_key = "doc_id"

retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
)
doc_ids = [str(uuid.uuid4()) for _ in docs]

**Wrap each summary as a `Document`**, tagging it with the matching `doc_id` in its metadata — this is what makes the summary→full-document link retrievable later.

In [ ]:
summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(summaries)
]

**Populate both stores:**
- `retriever.vectorstore.add_documents(summary_docs)` — embeds the summaries into `Chroma`.
- `retriever.docstore.mset(...)` — saves each full original document into the byte store, keyed by its `doc_id`.

In [ ]:
retriever.vectorstore.add_documents(summary_docs)
retriever.docstore.mset(list(zip(doc_ids, docs)))

## 6. Retrieve documents based on query

In [ ]:
query = "What is LangSmith?"
sub_docs = vectorstore.similarity_search(query)
sub_docs[0]

In [ ]:
retrieved_docs = retriever.invoke(query)

In [ ]:
retrieved_docs[0].page_content[0:500]

In [ ]:
len(retrieved_docs[0].page_content)